# 03 — Evaluation
**DS593 Final Project**

Evaluates the RAG system against a hand-crafted golden test set of 55 factual
questions with exact verified answers pulled directly from scraped documents.

Tests **6 system variations**:
- 3 chunking strategies × 2 retrieval methods
  - Chunking: Large (2048 chars) | Small (512 chars) | Sentence (NLTK)
  - Retrieval: Semantic (vector similarity) | Keyword (BM25)
- Plus 1 baseline: vanilla GPT with no retrieval

Scoring: fuzzy string match — does the model's answer contain the correct fact?


## Setup: Install Dependencies

In [ ]:
!pip install -q langchain-community langchain-openai langchain langchain-text-splitters python-dotenv sentence-transformers chromadb pandas rapidfuzz rank-bm25
print("✓ Dependencies installed")


## Step 0: Configure API Key

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("OPENAI_API_KEY") and os.getenv("OPENAI_API_KEY").startswith("sk-"):
    print("✓ OpenAI API key configured")
else:
    raise ValueError("Missing OpenAI API key")


## Step 1: Imports

In [ ]:
from typing import List
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from rank_bm25 import BM25Okapi
from rapidfuzz import fuzz
import pandas as pd
import json
import os

print("✓ All imports successful")


## Step 2: Load Embedding Model

In [ ]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

print("✓ Embedding model loaded: text-embedding-3-small")


## Step 3: Load Existing Vector Databases

Load the three ChromaDB collections already built in 01_chunk_index.ipynb.
No re-indexing needed.


In [ ]:
CHROMA_DIRS = {
    "large":    "chroma_db",
    "small":    "chroma_db_small",
    "sentence": "chroma_db_sentence",
}

vector_dbs = {}
for name, path in CHROMA_DIRS.items():
    try:
        vdb = Chroma(
            collection_name="tos_documents",
            persist_directory=path,
            embedding_function=embedding_model,
        )
        count = vdb._collection.count()
        vector_dbs[name] = vdb
        print(f"  ✓ {name:10s}: {count} chunks  ({path})")
    except Exception as e:
        print(f"  ✗ {name:10s}: could not load — {e}")

print(f"\n✓ {len(vector_dbs)} vector databases loaded")


## Step 4: Build BM25 Index

BM25 is a keyword-based retrieval method (no vectors, no embeddings).
It ranks chunks by term frequency — good at finding exact numbers and names.
We build one BM25 index per chunking strategy using the same corpus.


In [ ]:
def build_bm25_index(chroma_dir):
    """Build a BM25 index from an existing ChromaDB collection."""
    vdb = Chroma(
        collection_name="tos_documents",
        persist_directory=chroma_dir,
        embedding_function=embedding_model,
    )

    # Pull all documents and metadata from the collection
    data      = vdb._collection.get(include=["documents", "metadatas"])
    docs      = data["documents"]
    metas     = data["metadatas"]

    # Tokenize for BM25
    tokenized = [doc.lower().split() for doc in docs]
    bm25      = BM25Okapi(tokenized)

    return bm25, docs, metas


print("Building BM25 indexes...")
bm25_indexes = {}
for name, path in CHROMA_DIRS.items():
    bm25, docs, metas = build_bm25_index(path)
    bm25_indexes[name] = {"bm25": bm25, "docs": docs, "metas": metas}
    print(f"  ✓ {name:10s}: {len(docs)} docs indexed")

print("\n✓ BM25 indexes built")


## Step 5: Initialize LLM

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
    max_tokens=300
)

print("✓ LLM initialized")


## Step 6: Define Retrieval + Query Functions

Two retrieval methods, same generation prompt for fair comparison.


In [ ]:
def semantic_retrieve(vector_db, query: str, company: str, k: int = 3) -> List[str]:
    """Semantic retrieval using vector similarity search."""
    results = vector_db.similarity_search(
        query,
        k=k,
        filter={"company": company}
    )
    return [doc.page_content for doc in results]


def bm25_retrieve(bm25_index, query: str, company: str, k: int = 3) -> List[str]:
    """Keyword retrieval using BM25 scoring, filtered by company."""
    tokens   = query.lower().split()
    scores   = bm25_index["bm25"].get_scores(tokens)
    docs     = bm25_index["docs"]
    metas    = bm25_index["metas"]

    # Filter to company, rank by BM25 score
    filtered = [
        (i, scores[i])
        for i, meta in enumerate(metas)
        if meta.get("company") == company
    ]
    filtered.sort(key=lambda x: x[1], reverse=True)
    return [docs[i] for i, _ in filtered[:k]]


def generate_answer(chunks: List[str], question: str, company: str) -> str:
    """Generate answer from retrieved chunks — same prompt for all systems."""
    context = "\n\n".join(chunks)
    prompt  = f"""Answer based on these documents:
{context}

Question: {question}

Answer:"""
    return llm.invoke(prompt).content


def baseline_query(question: str, company: str) -> str:
    """Vanilla LLM with no retrieval."""
    prompt = f"""Answer this question about {company}'s Terms of Service or Privacy Policy.
Be direct and specific. If you are not sure, say so.

Question: {question}

Answer:"""
    return llm.invoke(prompt).content


print("✓ All query functions ready")


## Step 7: Load Golden Test Set

55 factual questions with exact verified answers.


In [ ]:
df = pd.read_csv("golden_test_set.csv")

print(f"✓ Loaded {len(df)} questions")
print(f"  Companies: {df['company'].nunique()}")
print(f"\n  Distribution:")
print(df["company"].value_counts().to_string())
print()
print(df[["company", "question", "answer"]].head(8).to_string(index=False))


## Step 8: Scoring Function

Fuzzy string match — checks whether the model's answer contains the ground truth fact.
Handles minor phrasing differences like "30 days" vs "thirty days".


In [ ]:
def contains_answer(model_answer: str, ground_truth: str, threshold: int = 80) -> bool:
    """Check if model answer contains the ground truth fact."""
    model_lower = model_answer.lower()
    truth_lower = ground_truth.lower()

    # Exact substring check first
    if truth_lower in model_lower:
        return True

    # Fuzzy partial match for slight variations
    return fuzz.partial_ratio(truth_lower, model_lower) >= threshold


# Sanity checks
assert contains_answer("You have 30 days to opt out of arbitration", "30 days")
assert not contains_answer("PayPal gives you 180 days to dispute", "60 days")
assert contains_answer("The governing state is Texas", "texas")
print("✓ Scoring function ready")


## Step 9: Run Evaluation

6 RAG systems + 1 baseline = 7 API calls per question.
Expect ~8-10 minutes for 55 questions.


In [ ]:
SYSTEMS = [
    ("large_semantic",   lambda q, c: generate_answer(semantic_retrieve(vector_dbs["large"],    q, c), q, c)),
    ("large_bm25",       lambda q, c: generate_answer(bm25_retrieve(bm25_indexes["large"],       q, c), q, c)),
    ("small_semantic",   lambda q, c: generate_answer(semantic_retrieve(vector_dbs["small"],    q, c), q, c)),
    ("small_bm25",       lambda q, c: generate_answer(bm25_retrieve(bm25_indexes["small"],       q, c), q, c)),
    ("sentence_semantic",lambda q, c: generate_answer(semantic_retrieve(vector_dbs["sentence"], q, c), q, c)),
    ("sentence_bm25",    lambda q, c: generate_answer(bm25_retrieve(bm25_indexes["sentence"],    q, c), q, c)),
    ("baseline",         lambda q, c: baseline_query(q, c)),
]

# Store answers and correctness for each system
for name, _ in SYSTEMS:
    df[f"ans_{name}"]     = ""
    df[f"correct_{name}"] = False

for i, row in df.iterrows():
    q       = row["question"]
    company = row["company"]
    truth   = row["answer"]
    print(f"[{i+1}/{len(df)}] {company} — {q[:55]}...")

    for name, query_fn in SYSTEMS:
        answer = query_fn(q, company)
        df.at[i, f"ans_{name}"]     = answer
        df.at[i, f"correct_{name}"] = contains_answer(answer, truth)

print("\n✓ Evaluation complete")


## Step 10: Results — Overall Accuracy

In [ ]:
total = len(df)

print("=" * 55)
print(f"{'System':<22} {'Correct':>8} {'Accuracy':>10}")
print("=" * 55)
for name, _ in SYSTEMS:
    correct = df[f"correct_{name}"].sum()
    pct     = correct / total * 100
    bar     = "█" * int(pct / 5)
    print(f"  {name:<20} {correct:>5}/{total}   {pct:>5.1f}%  {bar}")
print("=" * 55)


## Step 11: Results by Company

In [ ]:
rows = []
for name, _ in SYSTEMS:
    for company in df["company"].unique():
        sub = df[df["company"] == company]
        rows.append({
            "system":  name,
            "company": company,
            "n":       len(sub),
            "correct": sub[f"correct_{name}"].sum(),
            "pct":     sub[f"correct_{name}"].mean() * 100,
        })

pivot = pd.DataFrame(rows).pivot(index="company", columns="system", values="pct").round(1)
print("Accuracy (%) by company and system:\n")
print(pivot.to_string())


## Step 12: Semantic vs BM25 — Head to Head

In [ ]:
print("Semantic vs BM25 by chunking strategy:\n")
print(f"{'Strategy':<12} {'Semantic':>10} {'BM25':>10} {'Winner':>10}")
print("-" * 45)
for strategy in ["large", "small", "sentence"]:
    sem_acc  = df[f"correct_{strategy}_semantic"].mean() * 100
    bm25_acc = df[f"correct_{strategy}_bm25"].mean() * 100
    winner   = "Semantic" if sem_acc >= bm25_acc else "BM25"
    print(f"  {strategy:<12} {sem_acc:>9.1f}% {bm25_acc:>9.1f}% {winner:>10}")


## Step 13: Inspect Failures — Best RAG vs Baseline

In [ ]:
# Find the best performing RAG system
best_system = max(
    [name for name, _ in SYSTEMS if name != "baseline"],
    key=lambda n: df[f"correct_{n}"].sum()
)
print(f"Best RAG system: {best_system}\n")

print(f"Questions where {best_system} succeeded but baseline FAILED:")
print("=" * 70)
wins = df[(df[f"correct_{best_system}"] == True) & (df["correct_baseline"] == False)]
for _, row in wins.iterrows():
    print(f"\nCompany:  {row['company']}")
    print(f"Question: {row['question']}")
    print(f"Expected: {row['answer']}")
    print(f"RAG got:  {row[f'ans_{best_system}'][:200]}")
    print(f"Baseline: {row['ans_baseline'][:200]}")


## Step 14: Save Results

In [ ]:
df.to_csv("evaluation_results.csv", index=False)
print("✓ Saved to evaluation_results.csv\n")

best_acc = max(df[f"correct_{name}"].mean() * 100 for name, _ in SYSTEMS if name != "baseline")
base_acc = df["correct_baseline"].mean() * 100

print("=" * 55)
print("FINAL SUMMARY")
print("=" * 55)
print(f"  Total questions:   {total}")
print(f"  Best RAG accuracy: {best_acc:.1f}%  ({best_system})")
print(f"  Baseline accuracy: {base_acc:.1f}%")
print(f"  RAG improvement:   +{best_acc - base_acc:.1f}%")
